In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import  pandas as pd 
import json 
import os
from glob import glob
import seaborn as sns 
import numpy as np 
import re
import tikzplotly
import plotly.express as px
from IPython.display import display
from PIL import Image
import matplotlib as mpl
import matplotlib.pyplot as plt 
import plotly 
import plotly.graph_objects as go

from IPython.display import IFrame

from utils.benchmark import * 

In [2]:
notebook_name="01-c5.large-geobft-spread-over-azs"
os.makedirs(f"outputs/{notebook_name}", exist_ok=True)


import zipfile
import datetime

timestamp=datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

with zipfile.ZipFile(f'outputs/{notebook_name}/notebook_files_{timestamp}.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(f'outputs/{notebook_name}'):
        for file in files:
            if not file.endswith('.zip'):
                zipf.write(os.path.join(root, file), 
                           os.path.relpath(os.path.join(root, file), 
                                           os.path.join(f'outputs/{notebook_name}', '..')))
                os.remove(os.path.join(root, file))


In [3]:
# folder="../../aws/benchmark/out/modubft/full_spread_modubft_with_bytes_sent/" 
folder = "../../aws/benchmark/out/geobft/20251025-224449-large/"
os.listdir(folder)

['20251025224842-cb1-v512',
 '20251025225025-cb1-v4096',
 '20251025225206-cb16-v512',
 '20251025225351-cb16-v4096',
 '20251025225531-cb64-v512',
 '20251025225709-cb64-v4096',
 '20251025225844-cb128-v512',
 '20251025230018-cb128-v4096',
 '20251025231051-cb1-v512',
 '20251025231157-cb1-v4096',
 '20251025231257-cb16-v512',
 '20251025231359-cb16-v4096',
 '20251025231501-cb64-v512',
 '20251025231602-cb64-v4096',
 '20251025231705-cb128-v512',
 '20251025231806-cb128-v4096',
 '20251025232648-cb1-v512',
 '20251025232756-cb1-v4096',
 '20251025232901-cb16-v512',
 '20251025233009-cb16-v4096',
 '20251025233115-cb64-v512',
 '20251025233221-cb64-v4096',
 '20251025233327-cb128-v512',
 '20251025233429-cb128-v4096',
 '20251025234238-cb1-v512',
 '20251025234327-cb1-v4096',
 '20251025234417-cb16-v512',
 '20251025234506-cb16-v4096',
 '20251025234553-cb64-v512',
 '20251025234641-cb64-v4096',
 '20251025234730-cb128-v512',
 '20251025234817-cb128-v4096',
 '20251025235437-cb1-v512',
 '20251025235523-cb1-v4096',

In [4]:
benchmarks = glob(f"{folder}*/")
c5axlarge_folder="../../aws/benchmark/out/geobft/final/"
# benchmarks = benchmarks + glob(f"{c5axlarge_folder}*/")
benchmarks

['../../aws/benchmark/out/geobft/20251025-224449-large/20251025224842-cb1-v512/',
 '../../aws/benchmark/out/geobft/20251025-224449-large/20251025225025-cb1-v4096/',
 '../../aws/benchmark/out/geobft/20251025-224449-large/20251025225206-cb16-v512/',
 '../../aws/benchmark/out/geobft/20251025-224449-large/20251025225351-cb16-v4096/',
 '../../aws/benchmark/out/geobft/20251025-224449-large/20251025225531-cb64-v512/',
 '../../aws/benchmark/out/geobft/20251025-224449-large/20251025225709-cb64-v4096/',
 '../../aws/benchmark/out/geobft/20251025-224449-large/20251025225844-cb128-v512/',
 '../../aws/benchmark/out/geobft/20251025-224449-large/20251025230018-cb128-v4096/',
 '../../aws/benchmark/out/geobft/20251025-224449-large/20251025231051-cb1-v512/',
 '../../aws/benchmark/out/geobft/20251025-224449-large/20251025231157-cb1-v4096/',
 '../../aws/benchmark/out/geobft/20251025-224449-large/20251025231257-cb16-v512/',
 '../../aws/benchmark/out/geobft/20251025-224449-large/20251025231359-cb16-v4096/',


In [5]:




throughputs = process_throughput_benchmarks(benchmarks)
throughputs["benchmark"] = throughputs["benchmark"].apply(lambda s: "c5a.large" if "large" in s else "c5a.xlarge")


        
throughputs = add_combined_column(throughputs, ['number_of_clusters', 'vallen',"benchmark"], 'num_clusters_vallen')
throughputs["cluster_num_peers"] = throughputs["num_peers"] /  throughputs["number_of_clusters"] 
def generate_throughput_plot(throughputs, num_peers):
    xs = []
    ys = []
    cats = []

    for (group, df) in throughputs.query("cluster_num_peers == @num_peers").groupby("num_clusters_vallen"):
        num_clusters_vallen = group
        sorteddf = df.sort_values(by=["cluster_batch_size", "benchmark", "cluster_num_peers"])
        cats.append(num_clusters_vallen)
        xs.append(sorteddf["cluster_batch_size"].tolist())
        ys.append(sorteddf["throughput"].tolist())

    tikz_plot_throughput = TikzPlotGenerator(
        xs=xs,
        ys=ys,
        cat=cats,
        xlabel="Cluster Batch Size",
        ylabel="Throughput (ops/sec)",
        filename=f"outputs/{notebook_name}/geobft-spread-over-azs-throughput-{num_peers}.tex"
    )
    tikz_plot_throughput.save()
    tikz_plot_throughput.compile(output_dir=f"outputs/{notebook_name}/")

    return tikz_plot_throughput

generate_throughput_plot(throughputs, 6)
generate_throughput_plot(throughputs, 3)


[np.int64(1), np.int64(16), np.int64(64), np.int64(128)]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode

(./outputs/01-c5.large-geobft-spread-over-azs/geobft-spread-over-azs-throughput
-6.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-dist/tex/latex/s

In [6]:
throughputs = process_throughput_benchmarks(benchmarks)
throughputs["benchmark"] = throughputs["benchmark"].apply(lambda s: "c5a.large" if "large" in s else "c5a.xlarge")

throughputs["cluster_num_peers"] = throughputs["num_peers"] /  throughputs["number_of_clusters"] 

        
throughputs = add_combined_column(throughputs, ["cluster_num_peers",'number_of_clusters', 'vallen',"benchmark"], 'num_clusters_vallen')

throughputs["f"] = np.floor((throughputs["cluster_num_peers"]-1)/3)
throughputs["kf"] = throughputs["f"] * throughputs["number_of_clusters"]


def kf_throughput(throughputs,cluster_batch_size):
        
    xs = [] 
    ys = []
    cats = []


    for (group, df) in throughputs.query("cluster_batch_size == @cluster_batch_size").groupby("num_clusters_vallen"):
        num_clusters_vallen = group
        sorteddf = df.sort_values(by=["kf", "benchmark", "cluster_num_peers"])
        cats.append(num_clusters_vallen)
        xs.append(sorteddf["kf"].tolist())
        ys.append(sorteddf["throughput"].tolist())

    tikz_plot_throughput = TikzPlotGenerator(
    xs=xs,
    ys=ys,
    cat=cats,
    xlabel="kf (k * f) ",
    ylabel="Throughput (ops/sec)",
    filename=f"outputs/{notebook_name}/geobft-spread-over-azs-throughput-kf-{cluster_batch_size}.tex"
)
    tikz_plot_throughput.save()
    tikz_plot_throughput.compile(output_dir=f"outputs/{notebook_name}/")

kf_throughput(throughputs, 1)
kf_throughput(throughputs, 64)
kf_throughput(throughputs, 128)


[np.float64(0.0), np.float64(2.0), np.float64(4.0), np.float64(6.0)]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode

(./outputs/01-c5.large-geobft-spread-over-azs/geobft-spread-over-azs-throughput
-kf-1.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-d

In [7]:


processed_cpu_usage = process_cpu_usage(benchmarks)

role2_processed_cpu_usage = processed_cpu_usage.query("role == 2")
role2_cpu_stats = pd.concat([role2_processed_cpu_usage.groupby("benchmark").median().reset_index().assign(agg="median")])

role1_processed_cpu_usage = processed_cpu_usage.query("role == 1")
role1_cpu_stats = pd.concat([role1_processed_cpu_usage.groupby("benchmark").median().reset_index().assign(agg="median")])

        
role2_cpu_stats["benchmark"] = role2_cpu_stats["benchmark"].apply(lambda s: "c5a.large" if "large" in s else "c5a.xlarge")
role1_cpu_stats["benchmark"] = role1_cpu_stats["benchmark"].apply(lambda s: "c5a.large" if "large" in s else "c5a.xlarge")

role2_cpu_stats = add_combined_column(role2_cpu_stats, ['number_of_clusters', 'vallen',"benchmark","agg"], 'num_clusters_vallen')
role1_cpu_stats = add_combined_column(role1_cpu_stats, ['number_of_clusters', 'vallen',"benchmark","agg"], 'num_clusters_vallen')

role2_cpu_stats["cluster_num_peers"] = role2_cpu_stats["num_peers"] /  role2_cpu_stats["number_of_clusters"] 
role1_cpu_stats["cluster_num_peers"] = role1_cpu_stats["num_peers"] /  role1_cpu_stats["number_of_clusters"]


def sanitize_legend_entry(entry):
    """Escape LaTeX special characters in legend entries."""
    # Convert to string if not already
    entry = str(entry)
    # Escape special LaTeX characters
    replacements = {
        '_': r'\_',
        '%': r'\%',
        '$': r'\$',
        '#': r'\#',
        '&': r'\&',
        '{': r'\{',
        '}': r'\}',
        '~': r'\textasciitilde{}',
        '^': r'\textasciicircum{}',
        '\\': r'\textbackslash{}',
    }
    for char, escaped in replacements.items():
        entry = entry.replace(char, escaped)
    return entry
def generate_cpu_usage_plot(cpu_stats, num_peers, role):
    xs = []
    ys = []
    cats = []

    for (group, df) in cpu_stats.query("cluster_num_peers == @num_peers and role == @role").groupby("num_clusters_vallen"):
        num_clusters_vallen = group
        sorteddf = df.sort_values(by=["cluster_batch_size", "benchmark", "cluster_num_peers"])
        cats.append(sanitize_legend_entry(num_clusters_vallen))
        xs.append(sorteddf["cluster_batch_size"].tolist())
        ys.append(sorteddf["cpu_usage"].tolist())

    tikz_plot_cpu_usage = TikzPlotGenerator(
        xs=xs,
        ys=ys,
        cat=cats,
        xlabel="Cluster Batch Size",
        ylabel="CPU Usage in \%",
        filename=f"outputs/{notebook_name}/geobft-spread-over-azs-cpu-usage-{num_peers}-{role}.tex"
    )
    tikz_plot_cpu_usage.save()
    tikz_plot_cpu_usage.compile(output_dir=f"outputs/{notebook_name}/")

    return tikz_plot_cpu_usage

generate_cpu_usage_plot(role2_cpu_stats, 6, 2)
generate_cpu_usage_plot(role2_cpu_stats, 3, 2)
generate_cpu_usage_plot(role1_cpu_stats, 6, 1)
generate_cpu_usage_plot(role1_cpu_stats, 3, 1)



<>:57: SyntaxWarning: invalid escape sequence '\%'
<>:57: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipykernel_1546/3631122089.py:57: SyntaxWarning: invalid escape sequence '\%'
  ylabel="CPU Usage in \%",
/home/andre/miniconda3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/andre/miniconda3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/andre/miniconda3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/andre/miniconda3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/home/andre/miniconda3/lib/python3.13/site-packages/nump

[np.float64(1.0), np.float64(16.0), np.float64(64.0), np.float64(128.0)]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode

(./outputs/01-c5.large-geobft-spread-over-azs/geobft-spread-over-azs-cpu-usage-
6-2.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf

In [8]:

num_peers = throughputs["num_peers"]/throughputs["number_of_clusters"]
throughputs["cluster_num_peers"] = num_peers
throughput_in_a_cluster = throughputs["throughput"]/throughputs["number_of_clusters"]

preprepare_messages = throughput_in_a_cluster * num_peers
prepare_messages = throughput_in_a_cluster * num_peers 
commit_messages = throughput_in_a_cluster * (num_peers-1) 

cluster_preprepare = (throughput_in_a_cluster / throughputs["cluster_batch_size"]) * num_peers 
cluster_prepare = (throughput_in_a_cluster / throughputs["cluster_batch_size"]) * (np.floor((throughputs["number_of_clusters"] - 1 ) / 3 ) +1 )
cluster_commit = (throughput_in_a_cluster / throughputs["cluster_batch_size"]) * (throughputs["number_of_clusters"]-1) * num_peers
throughputs["total_messages"] = preprepare_messages + prepare_messages + commit_messages + cluster_preprepare + cluster_prepare + cluster_commit
throughputs["total_messages"] = throughputs["total_messages"] / 10 

add_combined_column(throughputs,["vallen","number_of_clusters","benchmark"],"num_clusters_vallen")

# fig = go.Figure() 

# for data in throughputs.groupby("vallen"): 
#     vallen, df = data
#     fig.add_trace(go.Scatter(
#         x=df["num_peers"],
#         y=df["total_messages"],
#         mode="lines+markers",
#         name=f"Vallen={vallen}"
#     ))
# fig.update_layout(
#     title="Total Messages Sent vs Number of Peers",
#     xaxis_title="Number of Peers",
#     yaxis_title="Total Messages Sent (messages/sec)"
# )
# fig.show()

plot_tikz(
    throughputs,
    x_col="cluster_batch_size",
    y_col="total_messages",
    cat_col="num_clusters_vallen",
    filename=f"outputs/{notebook_name}/geobft-spread-over-azs-leader-messages.tex",
    xlabel="Cluster Batch Size",
    ylabel="Total Messages Sent (messages/sec)",
    output_dir=f"outputs/{notebook_name}/"
)

plot_tikz(
    throughputs.query("cluster_batch_size == 128"),
    x_col="cluster_num_peers",
    y_col="total_messages",
    cat_col="num_clusters_vallen",
    filename=f"outputs/{notebook_name}/geobft-spread-over-azs-leader-messages-cluster-num-peers-128.tex",
    xlabel="Number of Peers per Cluster",
    ylabel="Total Messages Sent (messages/sec)",
    output_dir=f"outputs/{notebook_name}/"
)
plot_tikz(
    throughputs.query("cluster_batch_size == 1"),
    x_col="cluster_num_peers",
    y_col="total_messages",
    cat_col="num_clusters_vallen",
    filename=f"outputs/{notebook_name}/geobft-spread-over-azs-leader-messages-cluster-num-peers-1.tex",
    xlabel="Number of Peers per Cluster",
    ylabel="Total Messages Sent (messages/sec)",
    output_dir=f"outputs/{notebook_name}/"
)


[np.int64(1), np.int64(16), np.int64(64), np.int64(128)]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode

(./outputs/01-c5.large-geobft-spread-over-azs/geobft-spread-over-azs-leader-mes
sages.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-dist/tex/late

0

In [9]:
bytes_sent = process_bytes_sent(benchmarks)
bytes_sent["benchmark"] = bytes_sent["benchmark"].apply(lambda s: "c5a.large" if "large" in s else "c5a.xlarge")

In [10]:
bytes_sent_in_gbps = (bytes_sent.groupby(["num_peers","vallen","role","number_of_clusters","cluster_batch_size","benchmark"])[["bytes_sent"]].sum() * 8 * 10**(-9) * 15**(-1)).reset_index()
bytes_sent_in_gbps["bytes_sent"] = bytes_sent_in_gbps["bytes_sent"] / bytes_sent_in_gbps["number_of_clusters"]
bytes_sent_in_gbps = add_combined_column(bytes_sent_in_gbps, ['number_of_clusters', 'vallen',"benchmark"], 'num_clusters_vallen')
bytes_sent_in_gbps["cluster_num_peers"] = bytes_sent_in_gbps["num_peers"] /  bytes_sent_in_gbps["number_of_clusters"] 

for (group,bytes_sent_in_gbps) in bytes_sent_in_gbps.groupby(['cluster_num_peers']): 
    cluster_num_peers = group[0]

    bytes_sent_role2 = bytes_sent_in_gbps.query("role == 2")
    bytes_sent_role1 = bytes_sent_in_gbps.query("role == 1")

    plot_tikz(
        bytes_sent_role2,
        x_col="cluster_batch_size",
        y_col="bytes_sent",
        cat_col="num_clusters_vallen",
        filename=f"outputs/{notebook_name}/geobft-spread-over-azs-bytes-sent-role2-gbps-{cluster_num_peers}.tex",
        xlabel="Cluster Batch Size",
        ylabel="Bytes Sent (Gbps)",
        output_dir=f"outputs/{notebook_name}/"
    )

    plot_tikz(
        bytes_sent_role1,
        x_col="cluster_batch_size", 
        y_col="bytes_sent",
        cat_col="num_clusters_vallen",
        filename=f"outputs/{notebook_name}/geobft-spread-over-azs-bytes-sent-role1-gbps-{cluster_num_peers}.tex",
        xlabel="Cluster Batch Size",    
        ylabel="Bytes Sent (Gbps)",
        output_dir=f"outputs/{notebook_name}/"
        )
    

[np.int64(1), np.int64(16), np.int64(64), np.int64(128)]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode

(./outputs/01-c5.large-geobft-spread-over-azs/geobft-spread-over-azs-bytes-sent
-role2-gbps-3.0.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-dis

In [11]:
bytes_sent = add_combined_column(bytes_sent, ['num_peers', 'vallen','number_of_clusters',"cluster_batch_size","benchmark"], 'num_peers_vallen')
bytes_sent["cluster_num_peers"] = bytes_sent["num_peers"] /  bytes_sent["number_of_clusters"] 

# role_df = get_role_df(bytes_sent, role=2)
# role_df = add_combined_column(role_df, ['num_peers', 'vallen','number_of_clusters',"cluster_batch_size"], 'num_peers_vallen')
# role_df["cluster_num_peers"] = role_df["num_peers"] /  role_df["number_of_clusters"] 




In [12]:
bytes_sent = process_bytes_sent(benchmarks)
bytes_sent["benchmark"] = bytes_sent["benchmark"].apply(lambda s: "c5a.large" if "large" in s else "c5a.xlarge")
bytes_sent= bytes_sent.query("cluster_batch_size == 128")
bytes_sent["cluster_num_peers"] = bytes_sent["num_peers"] /  bytes_sent["number_of_clusters"] 
for (group,df) in bytes_sent.groupby(['cluster_num_peers']): 
    # display(df)
    # continue
    role_df = get_role_df(df, role=2)
    role_df = add_combined_column(role_df, ['vallen','number_of_clusters',"benchmark"], 'num_peers_vallen')
    role_df_complete = complete_multiindex(role_df, ['typ', 'vallen','number_of_clusters',"num_peers_vallen"])

    role_df_complete = add_percent_column(role_df_complete, 'num_peers_vallen', 'bytes_sent', 'bytes_sent_percent')
    cluster_num_peers = group[0]
    print(f"Cluster Num Peers: {cluster_num_peers}")
    
     


    plot_tikz(
        role_df_complete,
        x_col="num_peers_vallen",
        y_col="bytes_sent_percent",
        cat_col="typ",
        filename=f"outputs/{notebook_name}/geobft-spread-over-azs-bytes-sent-role2-{cluster_num_peers}.tex",
        xlabel="(Number of Peers, Vallen)",
        ylabel="Bytes Sent Percent",
        bar=True,
        stack=True,
        symbolic_x=True,
        sort_x_key=lambda x: (int(x.split(",")[0][1:]),int(x.split(",")[1]),str(x.split(",")[2:-1])),
        output_dir=f"outputs/{notebook_name}/"
        
    )
     

    role_df = get_role_df(df, role=1)
    role_df = add_combined_column(role_df, ['vallen','number_of_clusters',"benchmark"], 'num_peers_vallen')
    role_df_complete = complete_multiindex(role_df, ['typ', 'vallen','number_of_clusters',"num_peers_vallen"])

    role_df_complete = add_percent_column(role_df_complete, 'num_peers_vallen', 'bytes_sent', 'bytes_sent_percent')
    plot_tikz(
        role_df_complete,
        x_col="num_peers_vallen",
        y_col="bytes_sent_percent",
        cat_col="typ",
        filename=f"outputs/{notebook_name}/geobft-spread-over-azs-bytes-sent-role1-{cluster_num_peers}.tex",
        xlabel="(Number of Peers, Vallen)",
        ylabel="Bytes Sent Percent",
        bar=True,
        stack=True,
        symbolic_x=True,
        sort_x_key=lambda x: (int(x.split(",")[0][1:]),int(x.split(",")[1]),str(x.split(",")[2:-1])),
        output_dir=f"outputs/{notebook_name}/"
    )


/mnt/c/Users/Andre/OneDrive/Cloud-Native Byzantine Consensus/code/CloudModuBFT/src/evaluation/notebooks/utils/benchmark.py:364: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[new_col] = df[cols].apply(


Cluster Num Peers: 3.0
[np.str_('(512,2,c5a.large)'), np.str_('(512,4,c5a.large)'), np.str_('(512,6,c5a.large)'), np.str_('(4096,2,c5a.large)'), np.str_('(4096,4,c5a.large)'), np.str_('(4096,6,c5a.large)')]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode

(./outputs/01-c5.large-geobft-spread-over-azs/geobft-spread-over-azs-bytes-sent
-role2-3.0.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/te

/mnt/c/Users/Andre/OneDrive/Cloud-Native Byzantine Consensus/code/CloudModuBFT/src/evaluation/notebooks/utils/benchmark.py:364: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[new_col] = df[cols].apply(



Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cfg)
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2023/05/17 v1.4n Standard LaTeX document class
(/usr/share/texlive/texmf-dist/tex/latex/base/size10.clo))
(/usr/share/texlive/texmf-dist/tex/latex/pgf/frontendlayer/tikz.sty
(/usr/share/texlive/texmf-dist/tex/latex/pgf/basiclayer/pgf.sty
(/usr/share/texlive/texmf-dist/tex/latex/pgf/utilities/pgfrcs.sty
(/usr/share/texlive/

/mnt/c/Users/Andre/OneDrive/Cloud-Native Byzantine Consensus/code/CloudModuBFT/src/evaluation/notebooks/utils/benchmark.py:364: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[new_col] = df[cols].apply(


entering extended mode

(./outputs/01-c5.large-geobft-spread-over-azs/geobft-spread-over-azs-bytes-sent
-role2-6.0.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cfg)
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2023/05/17 v1.4n Standard LaTeX document class
(/usr/share/texlive/tex

/mnt/c/Users/Andre/OneDrive/Cloud-Native Byzantine Consensus/code/CloudModuBFT/src/evaluation/notebooks/utils/benchmark.py:364: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[new_col] = df[cols].apply(


entering extended mode

(./outputs/01-c5.large-geobft-spread-over-azs/geobft-spread-over-azs-bytes-sent
-role1-6.0.tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cfg)
(/usr/share/texlive/texmf-dist/tex/latex/base/article.cls
Document Class: article 2023/05/17 v1.4n Standard LaTeX document class
(/usr/share/texlive/tex

In [13]:
res = {}
array_length = 1000
throughputs = process_throughput_benchmarks(benchmarks)
for benchmark in benchmarks: 

    received = np.zeros(array_length)
    response = np.zeros(array_length)
    latency = np.zeros(array_length)
    num_leaders = 0 
    for log in glob(f"{benchmark}/*.log"): 
        
        with open(log, 'r') as f:
            lines = [line for line in f.read().splitlines() if "client request" in line]

        if len(lines) != 2*array_length : 
            continue
        if lines : 
           
            for line in lines : 
                nanoseconds = line.split(" ")[-1]
                if "Received" in line: 
                    rc = int(line.split(" ")[3])
                    # print(line)
                    received[rc] = int(nanoseconds)
                    rc += 1
                elif "Response" in line: 
                    rp = int(line.split(" ")[3])
                    response[rp] = int(nanoseconds)
                    rp += 1
                else: 
                    raise ValueError("Unexpected line")
            
            latency += response - received
            num_leaders+=1
    latency = latency / num_leaders
    latency_ms = latency * 1e-6
    max_latency = np.max(latency_ms)
    avg_latency = np.median(latency_ms)
    min_latency = np.min(latency_ms)
    throughputs.loc[throughputs['benchmark'] == benchmark, 'latency_ms'] = avg_latency
    throughputs.loc[throughputs['benchmark'] == benchmark, 'max_latency_ms'] = max_latency
    throughputs.loc[throughputs['benchmark'] == benchmark, 'min_latency_ms'] = min_latency
    
    if min_latency < 0 : 
        raise ValueError("Negative latency detected")
    print(f"Benchmark: {benchmark}, Max Latency: {max_latency}, Avg Latency: {avg_latency}, Min Latency: {min_latency}")
        

Benchmark: ../../aws/benchmark/out/geobft/20251025-224449-large/20251025224842-cb1-v512/, Max Latency: 1534.971136, Avg Latency: 921.6478933333333, Min Latency: 141.19381333333334
Benchmark: ../../aws/benchmark/out/geobft/20251025-224449-large/20251025225025-cb1-v4096/, Max Latency: 1841.3536426666667, Avg Latency: 1120.5284906666666, Min Latency: 198.858624
Benchmark: ../../aws/benchmark/out/geobft/20251025-224449-large/20251025225206-cb16-v512/, Max Latency: 212.82338133333334, Avg Latency: 140.15276799999998, Min Latency: 64.909824
Benchmark: ../../aws/benchmark/out/geobft/20251025-224449-large/20251025225351-cb16-v4096/, Max Latency: 471.5715413333333, Avg Latency: 295.02193066666666, Min Latency: 113.57414399999999
Benchmark: ../../aws/benchmark/out/geobft/20251025-224449-large/20251025225531-cb64-v512/, Max Latency: 154.803328, Avg Latency: 114.61220266666666, Min Latency: 57.13928533333333
Benchmark: ../../aws/benchmark/out/geobft/20251025-224449-large/20251025225709-cb64-v4096/

In [14]:
throughputs["benchmark"] = throughputs["benchmark"].apply(lambda s: "c5a.large" if "large" in s else "c5a.xlarge")
throughputs = add_combined_column(throughputs, ['number_of_clusters', 'vallen',"benchmark"], 'num_clusters_vallen')
throughputs["cluster_num_peers"] = throughputs["num_peers"] /  throughputs["number_of_clusters"]
def generate_latency_plot(throughputs, num_peers):
    xs = []
    ys = []
    cats = []

    for (group, df) in throughputs.query("cluster_num_peers == @num_peers").groupby("num_clusters_vallen"):
        num_clusters_vallen = group
        sorteddf = df.sort_values(by=["cluster_batch_size", "benchmark", "cluster_num_peers"])
        cats.append(num_clusters_vallen)
        xs.append(sorteddf["cluster_batch_size"].tolist())
        ys.append(sorteddf["latency_ms"].tolist())

    tikz_plot_latency = TikzPlotGenerator(
        xs=xs,
        ys=ys,
        cat=cats,
        xlabel="Cluster Batch Size",
        ylabel="Latency (ms)",
        filename=f"outputs/{notebook_name}/geobft-spread-over-azs-latency-{num_peers}.tex"
    )
    tikz_plot_latency.save()
    tikz_plot_latency.compile(output_dir=f"outputs/{notebook_name}/")

    return tikz_plot_latency

generate_latency_plot(throughputs, 6)
generate_latency_plot(throughputs, 3)

#     throughputs,
#     y_col="latency_ms",
#     x_col="num_peers",
#     cat_col="vallen",
#     filename=f"outputs/{notebook_name}/geobft-spread-over-azs-latency.tex",
#     xlabel="Number of Peers",
#     ylabel="Latency (ms)",
#     output_dir=f"outputs/{notebook_name}/"
# )

[np.int64(1), np.int64(16), np.int64(64), np.int64(128)]
This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode

(./outputs/01-c5.large-geobft-spread-over-azs/geobft-spread-over-azs-latency-6.
tex
LaTeX2e <2023-11-01> patch level 1
L3 programming layer <2024-01-22>
(/usr/share/texlive/texmf-dist/tex/latex/standalone/standalone.cls
Document Class: standalone 2022/10/10 v1.3b Class to compile TeX sub-files stan
dalone
(/usr/share/texlive/texmf-dist/tex/latex/tools/shellesc.sty)
(/usr/share/texlive/texmf-dist/tex/generic/iftex/ifluatex.sty
(/usr/share/texlive/texmf-dist/tex/generic/iftex/iftex.sty))
(/usr/share/texlive/texmf-dist/tex/latex/xkeyval/xkeyval.sty
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkeyval.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/xkvutils.tex
(/usr/share/texlive/texmf-dist/tex/generic/xkeyval/keyval.tex))))
(/usr/share/texlive/texmf-dist/tex/latex/stan

In [15]:
#rename every file in output_dir=f"outputs/{notebook_name}/"
import os
output_dir=f"outputs/{notebook_name}/"
for filename in os.listdir(output_dir):
    if filename.endswith(".tex") or filename.endswith(".pdf"):
        new_filename = filename.replace("geobft-spread-over-azs","geobft-spread-over-azs-c5a-large")
        os.rename(os.path.join(output_dir, filename), os.path.join(output_dir, new_filename))
        
# remove log and aux 
for filename in os.listdir(output_dir):
    if filename.endswith(".log") or filename.endswith(".aux"):
        os.remove(os.path.join(output_dir, filename))

In [16]:
timestamp=datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

with zipfile.ZipFile(f'outputs/{notebook_name}/notebook_files_{timestamp}.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(f'outputs/{notebook_name}'):
        for file in files:
            if not file.endswith('.zip'):
                zipf.write(os.path.join(root, file), 
                           os.path.relpath(os.path.join(root, file), 
                                           os.path.join(f'outputs/{notebook_name}', '..')))

In [19]:

s = ""
for filename in os.listdir(output_dir):
    if filename.endswith(".pdf"):
        tex_content = ""
        with open(os.path.join(output_dir, filename.replace(".pdf",".tex")), 'r') as f:
            for line in f:
                tex_content += "%" + line

        fig=r"""
            \begin{figure}[htbp]
            
            """+f"{tex_content}"+"""
                \centering
                \includegraphics{"""+f"figures/{notebook_name}/{filename}"+"""}
                \caption{fill}
                \label{fig:modubft-spread-over-azs-cpu-leader}
            \end{figure}
            """
        s+=fig+"\n\n"
        

with open(f"outputs/{notebook_name}/figures.tex", 'w') as f:
    f.write(s)


<>:13: SyntaxWarning: invalid escape sequence '\c'
<>:15: SyntaxWarning: invalid escape sequence '\c'
<>:13: SyntaxWarning: invalid escape sequence '\c'
<>:15: SyntaxWarning: invalid escape sequence '\c'
/tmp/ipykernel_1546/557275016.py:13: SyntaxWarning: invalid escape sequence '\c'
  \centering
/tmp/ipykernel_1546/557275016.py:15: SyntaxWarning: invalid escape sequence '\c'
  \caption{fill}
